In [ ]:
import pandas as pd
import numpy as np
import traceback
import os
import sys
import warnings
from datetime import datetime
import tkinter as tk
from tkinter import filedialog

# --- VERIFICACIÓN DE LIBRERÍAS ---
try:
    import xlwings as xw
except ImportError:
    print("ERROR CRÍTICO: Falta 'xlwings'. Ejecuta: pip install xlwings")
    sys.exit(1)

try:
    import xlsxwriter
except ImportError:
    print("ERROR CRÍTICO: Falta 'xlsxwriter'. Ejecuta: pip install xlsxwriter")
    sys.exit(1)

# Silenciar advertencias
warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURACIÓN
# =============================================================================
HOJA_COMANDAS_DESTINO = "Devoluciones"
HOJA_CAJA_ADICION_DESTINO = "Caja Adicion"
HOJA_CANCELADOS_PY_DESTINO = "CanceladosPY"
HOJA_PAGOS_MP_DESTINO = "Pagos MP" 
HOJA_NO_CONCILIADO_DESTINO = "No conciliado"

FILES_OUTPUT = {}

CONFIG_CONCILIACION = {
    
    'col_fecha_mp': 'FECHA DE ORIGEN',
    'col_monto_mp': 'VALOR DE LA COMPRA',
    'col_id_mp': 'ID DE OPERACIÓN EN MERCADO PAGO',
    'col_tipo_op_mp': 'TIPO DE OPERACIÓN',
    'col_fecha_py': 'Fecha del pedido',   # <--- NUEVO
    'col_monto_py': 'Total del pedido',   # <--- NUEVO
    'col_id_py': 'Nro de pedido', # <--- AGREGAR ESTA LÍNEA
    'col_fecha_sis': 'Fecha',
    'col_id_sis': 'ID de venta',
    'col_monto_sis': 'Monto Bruto pago',
    'col_propina_sis': 'Propina',         # <--- AGREGAR ESTA LÍNEA
    'col_a_pagar_sis': 'A Pagar',         # <--- AGREGAR ESTA LÍNEA
    'col_plataforma': "Medio de cobro",
    'col_turno': "TURNO",
    'val_mp': "Mercado Pago",
    'val_py_sis': "Pedidos Ya",
    'val_efectivo': "Efectivo",
    'val_cta_cte': "Cta Cte"
}

REGLAS_CONCILIACION = [
    ("R0: 1min / $1", 1, 1),        
    ("R1: 5min / $5", 5, 5),        
    ("R2: 30min / $5", 30, 5),
    ("R3: Mismo Día / $5", 9999, 5) 
]

KEYWORDS_EXCLUIR = ['interno', 'ingreso extra']

# =============================================================================
# FUNCIONES AUXILIARES
# =============================================================================

def seleccionar_archivo(titulo_ventana):
    print(f"--> Por favor, selecciona el archivo: {titulo_ventana}")
    root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
    ruta = filedialog.askopenfilename(title=f"SELECCIONA: {titulo_ventana}", filetypes=[("Archivos Excel", "*.xlsx;*.xls;*.xlsm;*.html")])
    root.destroy()
    if not ruta: sys.exit()
    print(f"    OK: {os.path.basename(ruta)}")
    return ruta

def seleccionar_carpeta(titulo_ventana):
    print(f"--> Por favor, selecciona la CARPETA DE DESTINO.")
    root = tk.Tk(); root.withdraw(); root.attributes('-topmost', True)
    ruta = filedialog.askdirectory(title=titulo_ventana)
    root.destroy()
    if not ruta: sys.exit()
    print(f"    OK: Carpeta {ruta}")
    return ruta

def parsear_fecha_mp_iso(series):
    s = series.astype(str).str.strip()
    s = s.str.slice(0, 19).str.replace("T", " ", regex=False)
    fechas = pd.to_datetime(s, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    mask_nulos = fechas.isna()
    if mask_nulos.any():
        fechas.loc[mask_nulos] = pd.to_datetime(s[mask_nulos], dayfirst=True, errors='coerce')
    return fechas

def normalizar_fecha_argentina(series):
    if pd.api.types.is_datetime64_any_dtype(series): return series
    s = series.astype(str).str.lower().str.strip()
    basura = ['"', "'", 'p. m.', 'a. m.', 'p.m.', 'a.m.', 'p m', 'a m', '.']
    for b in basura:
        if b in ['p. m.', 'p.m.', 'p m']: s = s.str.replace(b, 'pm', regex=False)
        elif b in ['a. m.', 'a.m.', 'a m']: s = s.str.replace(b, 'am', regex=False)
        else: s = s.str.replace(b, '', regex=False)
    return pd.to_datetime(s, dayfirst=True, errors='coerce')

def convertir_a_string_visual(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return series.dt.strftime('%d/%m/%Y %H:%M:%S').fillna('')
    return series

def formato_visual_columna(df, col_name):
    if col_name in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df[col_name]):
             try: df[col_name] = pd.to_datetime(df[col_name], dayfirst=True, errors='coerce')
             except: pass
        if pd.api.types.is_datetime64_any_dtype(df[col_name]):
            return df[col_name].dt.strftime('%d/%m/%Y %H:%M:%S').fillna('')
    return df[col_name]

def limpiar_monto_general(valor):
    """
    Lógica de conversión corregida para formato Argentino (Getnet/Sistema).
    Prioriza la coma como separador decimal.
    """
    if pd.isna(valor) or str(valor).strip() == '': return 0.0
    if isinstance(valor, (int, float)): return float(valor)
    
    s = str(valor).strip().replace('$', '').replace('ARS', '').replace(' ', '').replace('\xa0', '')
    
    if ',' in s:
        # Caso 79.200,08 -> Eliminamos el punto de miles, cambiamos coma por punto
        s = s.replace('.', '').replace(',', '.')
    else:
        # Caso 20.000 (sin coma) -> El punto suele ser separador de miles
        if '.' in s:
            partes = s.split('.')
            if len(partes) > 2 or len(partes[1]) == 3:
                s = s.replace('.', '')
            
    try: return float(s)
    except ValueError: return 0.0

def asignar_turno_desde_excel(df_reporte, col_fecha_dt, df_turnos_maestro, col_ap_maestro='Apertura_DT', col_ci_maestro='Cierre_DT'):
    """
    MODIFICADO: Ahora asigna TURNO y FECHA_TURNO (Fecha de Apertura del Turno).
    Si es un turno de madrugada (ej 03/01 02:00am que pertenece al turno del 02/01),
    FECHA_TURNO será 02/01.
    """
    if df_turnos_maestro.empty:
        df_reporte['TURNO'] = "Sin Turnos"
        df_reporte['FECHA_TURNO'] = pd.NaT
        return df_reporte
    
    # Asegurar formato fecha
    if not pd.api.types.is_datetime64_any_dtype(df_reporte[col_fecha_dt]):
          df_reporte[col_fecha_dt] = pd.to_datetime(df_reporte[col_fecha_dt], dayfirst=True, errors='coerce')
          
    try:
        if df_reporte[col_fecha_dt].dt.tz is not None:
            df_reporte[col_fecha_dt] = df_reporte[col_fecha_dt].dt.tz_localize(None)
    except: pass

    # Función interna para buscar
    def find_data_turno(transaction_time):
        if pd.isna(transaction_time): return "Fecha Nula", pd.NaT
        # Busca si la hora cae en algún turno
        mask = (transaction_time >= df_turnos_maestro[col_ap_maestro]) & \
               (transaction_time <= df_turnos_maestro[col_ci_maestro])
        matching = df_turnos_maestro.loc[mask]
        
        if not matching.empty: 
            # Retorna: (Nombre del Turno, Fecha de Apertura de ese Turno)
            return matching.iloc[0]['TURNO'], matching.iloc[0]['Fecha Apertura']
        
        # Si no encuentra turno, devuelve fecha real
        return "Fuera de turno", pd.to_datetime(transaction_time.date())

    # Aplicamos la búsqueda fila por fila
    resultados = df_reporte[col_fecha_dt].apply(find_data_turno)
    
    # Guardamos los dos datos en columnas separadas
    df_reporte['TURNO'] = [res[0] for res in resultados]
    df_reporte['FECHA_TURNO'] = [res[1] for res in resultados]
    
    # Aseguramos que sea fecha y no texto
    df_reporte['FECHA_TURNO'] = pd.to_datetime(df_reporte['FECHA_TURNO'], errors='coerce')

    return df_reporte

    

def calcular_mascara_exclusion(series_clasificacion):
    s = series_clasificacion.fillna('').astype(str).str.lower().str.strip()
    for char_old, char_new in [('á','a'), ('é','e'), ('í','i'), ('ó','o'), ('ú','u')]:
        s = s.str.replace(char_old, char_new, regex=False)
    return s.isin(KEYWORDS_EXCLUIR)

# =============================================================================
# FASE 1: PROCESAMIENTO
# =============================================================================
def limpiar_zonas_horarias_profundo(df):
    """
    Función auxiliar para eliminar zonas horarias de cualquier columna 
    y evitar que Excel falle al guardar.
    """
    for col in df.columns:
        # 1. Si es columna de fecha
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            try: df[col] = df[col].dt.tz_localize(None)
            except: pass
        
        # 2. Si es columna de texto/objeto (revisión celda por celda)
        elif df[col].dtype == 'object':
            try:
                df[col] = df[col].apply(lambda x: x.replace(tzinfo=None) if hasattr(x, 'tzinfo') and x.tzinfo is not None else x)
            except: pass
    return df

def procesar_pedidos_ya_retorno(ruta_input, df_turnos_maestro):
    print(f"    -> [DEBUG] Iniciando carga PedidosYa: {os.path.basename(ruta_input)}")
    
    if not ruta_input or not os.path.exists(ruta_input): 
        print("    [ERROR] Ruta de PedidosYa inválida.")
        return pd.DataFrame(), pd.DataFrame()

    df = pd.DataFrame()
    leido = False

    # --- INTENTO 1: LECTURA RÁPIDA (PANDAS) ---
    try:
        # Intentamos leer directo. Si el XML está sucio y es un EXE, esto fallará.
        df = pd.read_excel(ruta_input, header=1)
        leido = True
        print("    [OK] Leído con motor estándar.")
    except Exception as e:
        print(f"    [AVISO] Falló lectura estándar ({e}). Activando Plan B (Excel Real)...")

    # --- INTENTO 2: LECTURA ROBUSTA (XLWINGS - Plan B) ---
    # Esto simula abrir el archivo manualmente en Excel (que arregla el XML solo)
    if not leido:
        app = None
        try:
            import xlwings as xw
            app = xw.App(visible=False) # Abre Excel en segundo plano
            wb = app.books.open(ruta_input)
            hoja = wb.sheets[0]
            
            # Buscamos dónde arrancan los datos (Buscamos la palabra "Fecha" en las primeras filas)
            # PedidosYa suele tener encabezado en la fila 1 o 2.
            datos_dump = hoja.range('A1:Z5').value # Leemos un pedacito para investigar
            header_row = 0
            
            for i, fila in enumerate(datos_dump):
                fila_str = [str(x).lower() for x in fila if x is not None]
                # Palabras clave de PedidosYa
                if any("fecha del pedido" in s for s in fila_str) or any("total del pedido" in s for s in fila_str):
                    header_row = i + 1 # xlwings usa base 1
                    break
            
            if header_row == 0: header_row = 2 # Default si no encuentra nada
            
            print(f"    [INFO] Cabecera detectada en fila {header_row}")
            
            # Leemos la tabla completa usando la funcionalidad de Tabla de xlwings
            # options(pd.DataFrame) convierte directo a Pandas
            df = hoja.range(f'A{header_row}').options(pd.DataFrame, index=False, expand='table').value
            
            wb.close()
            leido = True
            print("    [SALVADO] Archivo recuperado usando Excel Real (xlwings).")
            
        except Exception as ex:
            print(f"    [ERROR CRITICO] Falló también el Plan B: {ex}")
            import traceback; traceback.print_exc()
            return pd.DataFrame(), pd.DataFrame()
        finally:
            # Asegurar que Excel se cierre pase lo que pase
            if app:
                try: app.quit()
                except: pass

    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    # --- LIMPIEZA Y PROCESAMIENTO (Igual que siempre) ---
    try:
        # Normalizar nombres de columnas
        df.columns = df.columns.astype(str).str.strip()
        
        # Limpieza de tipos numéricos
        cols_numericas = ['Total del pedido', 'Monto de pago', 'Comisión', 'Cargos']
        for col in cols_numericas:
            if col in df.columns: 
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        # Limpieza de fechas
        cols_fechas = ['Fecha del pedido', 'Aceptado en', 'Entregado el', 'Cancelado el']
        for col in cols_fechas:
            if col in df.columns: 
                df[col] = pd.to_datetime(df[col], errors='coerce')
        
        # Función auxiliar de limpieza de zonas (asegúrate que exista en tu código)
        try: df = limpiar_zonas_horarias_profundo(df)
        except: pass

        # Asignar Turnos
        if 'Fecha del pedido' in df.columns and not df_turnos_maestro.empty:
            df = asignar_turno_desde_excel(df, 'Fecha del pedido', df_turnos_maestro)

        # 1. Separar Cancelados
        if 'Estado del pedido' in df.columns:
            df_cancelados = df[df['Estado del pedido'].astype(str).str.upper() == 'CANCELADO'].copy()
        else:
            df_cancelados = pd.DataFrame()

        # 2. Generar Data Limpia
        if 'Forma de pago' in df.columns:
            df_limpio = df[df['Forma de pago'] == "Pago online"].copy()
        else:
            df_limpio = df.copy()

        # PREVENCIÓN DE ERROR 'KEYERROR: ESTADO'
        # Aseguramos que las columnas críticas existan sí o sí
        df_limpio['Estado'] = 'No Conciliado' 
        df_limpio['Tipo Match'] = ''
        
        return df_limpio, df_cancelados

    except Exception as e:
        print(f"    [ERROR PROCESANDO DATOS PY]: {e}")
        import traceback; traceback.print_exc()
        return pd.DataFrame(), pd.DataFrame()

def exportar_a_macro_simple(df, ruta_macro, nombre_hoja):
    """Exporta al archivo Macro (para los cancelados)"""
    print(f"    -> Exportando '{nombre_hoja}' al Macro...")
    if df.empty or not os.path.exists(ruta_macro): return

    app = xw.App(visible=False)
    try:
        wb = app.books.open(ruta_macro)
        try: ws = wb.sheets[nombre_hoja]; ws.clear()
        except: ws = wb.sheets.add(nombre_hoja)
        
        ws.range('A1').options(index=False).value = df
        ws.autofit()
        wb.save()
    except Exception as ex: print(f"Error xlwings ({nombre_hoja}): {ex}")
    finally: 
        try: wb.close(); app.quit()
        except: pass


def procesar_archivo_turnos(ruta_archivo):
    try: 
        # Leemos el archivo completo
        df = pd.read_excel(ruta_archivo, sheet_name=0)
    except: 
        return pd.DataFrame()
    
    # --- LIMPIEZA AGRESIVA INICIAL ---
    # Esto limpia las celdas apenas se leen del Excel original
    df = limpiar_zonas_horarias_profundo(df)

    # Normalizamos columnas base
    cols_fecha = ['Fecha Apertura', 'Fecha Cierre', 'Fecha Apertura MP', 'Fecha Cierre MP']
    for c in cols_fecha:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], dayfirst=True, errors='coerce')
            try: df[c] = df[c].dt.tz_localize(None).dt.normalize()
            except: df[c] = df[c].dt.normalize()

    if 'TURNO' in df.columns:
        df['TURNO'] = df['TURNO'].astype(str).str.strip().str.upper()
    
    columnas_req = ['Fecha Apertura', 'Hs Ap. Caja', 'Fecha Cierre', 'Hs Cierre Caja', 'TURNO']
    if not all(col in df.columns for col in columnas_req): 
        print("Faltan columnas requeridas en el archivo de Turnos.")
        return pd.DataFrame()
    
    try:
        # Armado de Fechas
        ap_str = df['Fecha Apertura'].dt.strftime('%Y-%m-%d') + ' ' + df['Hs Ap. Caja'].astype(str)
        ci_str = df['Fecha Cierre'].dt.strftime('%Y-%m-%d') + ' ' + df['Hs Cierre Caja'].astype(str)
        
        df['Apertura_DT'] = pd.to_datetime(ap_str, errors='coerce')
        df['Cierre_DT'] = pd.to_datetime(ci_str, errors='coerce')

        if 'Fecha Apertura MP' in df.columns and 'Hs Ap. Caja MP' in df.columns:
            ap_mp_str = df['Fecha Apertura MP'].dt.strftime('%Y-%m-%d') + ' ' + df['Hs Ap. Caja MP'].astype(str)
            ci_mp_str = df['Fecha Cierre MP'].dt.strftime('%Y-%m-%d') + ' ' + df['Hs Cierre Caja MP'].astype(str)
            df['Apertura_MP_DT'] = pd.to_datetime(ap_mp_str, errors='coerce')
            df['Cierre_MP_DT'] = pd.to_datetime(ci_mp_str, errors='coerce')
        else:
            df['Apertura_MP_DT'] = df['Apertura_DT']
            df['Cierre_MP_DT'] = df['Cierre_DT']

        # --- LIMPIEZA AGRESIVA FINAL ---
        # Volvemos a limpiar antes de devolver el DataFrame para asegurar
        df = limpiar_zonas_horarias_profundo(df)

        return df
    except Exception as e: 
        print(f"Error interno procesando turnos: {e}")
        return pd.DataFrame()
    
def procesar_caja_adicion(ruta_input, ruta_output_xlsm, nombre_hoja_destino, df_turnos_maestro):
    print(f"    -> Procesando Caja Adición hacia '{nombre_hoja_destino}'...")
    if not os.path.exists(ruta_input): return

    try:
        df = pd.read_excel(ruta_input, sheet_name="Hoja1")

        df['Fecha Modificación'] = df['Fecha Modificación'].astype(str).str.replace('"', '', regex=False).str.replace('p. m.', 'PM', regex=False).str.replace('a. m.', 'AM', regex=False).str.upper()
        df['Fecha Modificación'] = pd.to_datetime(df['Fecha Modificación'], dayfirst=True, errors='coerce')
        
        # Caja usa turno PRINCIPAL
        if not df_turnos_maestro.empty:
            df = asignar_turno_desde_excel(df, 'Fecha Modificación', df_turnos_maestro)
        else:
            df['TURNO'] = "Sin Data Turnos"

        df['Fecha'] = df['Fecha Modificación']
        df['Hora'] = df['Fecha Modificación']
        
        df['Fecha Contable'] = pd.to_datetime(df['Fecha Contable'], dayfirst=True, errors='coerce')
        df['Fecha Pago/Venc.'] = pd.to_datetime(df['Fecha Pago/Venc.'], dayfirst=True, errors='coerce')
        
        cols_numericas = ['Monto', 'Monto EDIT.', 'Q.REC', 'Q.FAC', 'PRECIO']
        for col in cols_numericas:
            if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')

        movimientos_validos = ['Egreso de Dinero', 'Ingreso de Dinero']

        if 'Forma de Pago' in df.columns:
            df_filtrado = df[
                (df['Origen'] == 'Caja') & 
                (df['Proveedor / Para'].isin(movimientos_validos)) &
                (df['Forma de Pago'].astype(str).str.strip().str.lower() == 'efectivo')
            ].copy()
        else:
            df_filtrado = df[
                (df['Origen'] == 'Caja') & 
                (df['Proveedor / Para'].isin(movimientos_validos))
            ].copy()

        columnas_finales = ["Fecha Contable", "Fecha Modificación", "Fecha", "Hora", "TURNO", "Origen", "Clase", "Proveedor / Para", "Monto", "Comentario", "Usuario", "Tipo", "Forma de Pago", "Fecha Pago/Venc.", "N.I.F.", "# Doc", "Cuenta Contable", "Monto EDIT.", "LIN", "Q.REC", "Q.FAC", "PRECIO"]
        df_final = df_filtrado[[c for c in columnas_finales if c in df_filtrado.columns]]

        app = xw.App(visible=False)
        try:
            wb = app.books.open(ruta_output_xlsm)
            try: ws = wb.sheets[nombre_hoja_destino]; ws.clear()
            except: ws = wb.sheets.add(nombre_hoja_destino)
            
            if not df_final.empty:
                ws.range('A1').options(index=False).value = df_final
                ws.autofit()
                
                last_row = len(df_final) + 1
                ws.range(f'B2:B{last_row}').number_format = 'dd/mm/yyyy hh:mm:ss'
                ws.range(f'C2:C{last_row}').number_format = 'dd/mm/yyyy'
                ws.range(f'D2:D{last_row}').number_format = 'hh:mm:ss'
                
            wb.save()
            print("    -> Caja Adición exportada.")
        except Exception as ex: print(f"Error xlwings en Caja Adicion: {ex}")
        finally: 
            try: wb.close(); app.quit()
            except: pass
            
    except Exception as e: print(f"Error procesando Caja Adicion: {e}")

def obtener_df_pagos_mp_negativos(ruta_input_mp, df_turnos_maestro):
    """
    PROCESO ESPECIAL: Usa las columnas SECUNDARIAS de MP para asignar el turno.
    """
    print(f"    -> Procesando Pagos MP (Negativos) con HORARIO MP EXTENDIDO...")
    if not os.path.exists(ruta_input_mp): return pd.DataFrame()

    try:
        df = pd.read_excel(ruta_input_mp)
        df.columns = df.columns.astype(str).str.strip().str.upper()
        col_fecha_mp = 'FECHA DE ORIGEN'
        col_id_mp = 'ID DE OPERACIÓN EN MERCADO PAGO'
        col_monto_mp = 'MONTO NETO DE LA OPERACIÓN QUE IMPACTÓ TU DINERO'

        if col_fecha_mp not in df.columns: return pd.DataFrame()

        df['FECHA_DT_TEMP'] = parsear_fecha_mp_iso(df[col_fecha_mp]) 
        
        # --- ASIGNACIÓN CON COLUMNAS SECUNDARIAS ---
        df = asignar_turno_desde_excel(df, "FECHA_DT_TEMP", df_turnos_maestro, 
                                       col_ap_maestro='Apertura_MP_DT', 
                                       col_ci_maestro='Cierre_MP_DT')
        
        df['Fecha y Hora'] = df['FECHA_DT_TEMP'] 
        df['Fecha'] = df['FECHA_DT_TEMP'].dt.normalize()
        df['Hora'] = df['FECHA_DT_TEMP'] 

        if col_monto_mp in df.columns: df[col_monto_mp] = pd.to_numeric(df[col_monto_mp], errors='coerce').fillna(0)
        
        df_negativos = df[df[col_monto_mp] < 0].copy()
        
        if df_negativos.empty: return pd.DataFrame()
        if col_id_mp in df_negativos.columns: df_negativos[col_id_mp] = df_negativos[col_id_mp].astype(str).str.replace(r'\.0$', '', regex=True)

        columnas_deseadas = ['Fecha y Hora', 'Fecha', 'Hora', col_id_mp, col_monto_mp, 'TURNO']
        return df_negativos[[c for c in columnas_deseadas if c in df_negativos.columns]]

    except Exception as e: 
        print(f"Error procesando Pagos MP: {e}")
        return pd.DataFrame()

def procesar_comandas(ruta_input, ruta_output_xlsm, nombre_hoja_destino, df_turnos_maestro):
    print(f"    -> Procesando Comandas hacia '{nombre_hoja_destino}'...")
    if not os.path.exists(ruta_input): return

    try:
        try: df = pd.read_excel(ruta_input, dtype=str)
        except:
            try: dfs = pd.read_html(ruta_input); df = max(dfs, key=len) if dfs else pd.DataFrame()
            except: return

        cols_actuales = [str(c).strip() for c in df.columns]
        if "ID Comanda" not in cols_actuales:
            for i in range(min(20, len(df))):
                fila = df.iloc[i].astype(str).tolist()
                if "ID Comanda" in fila:
                    df.columns = df.iloc[i]; df = df[i+1:].copy(); df.reset_index(drop=True, inplace=True); break
        df.columns = df.columns.astype(str).str.strip()
        
        if "Precios" in df.columns: df["Precios"] = df["Precios"].apply(limpiar_monto_general).fillna(0).astype('int64')
        if "ID Comanda" in df.columns: df["ID Comanda"] = df["ID Comanda"].astype(str).str.replace(r'\.0$', '', regex=True)

        col_fecha_pedido = "Fecha Hora pedido" 
        if "Hora pedido" in df.columns:
            df["Hora pedido DT"] = normalizar_fecha_argentina(df["Hora pedido"])
            df["Hora pedido"] = df["Hora pedido DT"].dt.strftime('%H:%M:%S').fillna('')
            df[col_fecha_pedido] = df["Hora pedido DT"]
        else: df[col_fecha_pedido] = pd.NaT

        # Comandas usan turno PRINCIPAL
        if not df_turnos_maestro.empty and col_fecha_pedido in df.columns:
            df = asignar_turno_desde_excel(df, col_fecha_pedido, df_turnos_maestro)
        else: df['TURNO'] = "Sin Data Fecha"
            
        cols_finales = ["ID Comanda", "TURNO", "Camarero Mesa", "Mesa", "Producto", "Precios", "Comentario", col_fecha_pedido, "Hora pedido", "Hora Anulación"]
        df_final = df[[c for c in cols_finales if c in df.columns]]

        app = xw.App(visible=False)
        try:
            wb = app.books.open(ruta_output_xlsm)
            try: ws = wb.sheets[nombre_hoja_destino]; ws.clear()
            except: ws = wb.sheets.add(nombre_hoja_destino)
            if not df_final.empty:
                ws.range('A1').options(index=False).value = df_final
                
                cols_validas = [c for c in cols_finales if c in df.columns]
                if col_fecha_pedido in cols_validas:
                    try:
                        idx = cols_validas.index(col_fecha_pedido)
                        letras = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
                        letra = letras[idx] if idx < 26 else "H"
                        last_row = len(df_final) + 1
                        ws.range(f'{letra}2:{letra}{last_row}').number_format = 'dd/mm/yyyy hh:mm:ss'
                    except: pass
                ws.autofit()
            wb.save()
            print("    -> Comandas exportadas.")
        except Exception as ex: print(f"Error xlwings: {ex}")
        finally: 
            try: wb.close(); app.quit()
            except: pass
    except Exception as e: print(f"Error procesando comandas: {e}")

# =============================================================================
# FASE 2: CONCILIACIÓN
# =============================================================================

def exportar_tablas_a_xlwings(tablas, ruta_macro, nombre_hoja):
    print(f"    -> Exportando tablas a MACRO ({nombre_hoja})...")
    if not os.path.exists(ruta_macro): return
    app = xw.App(visible=False)
    try:
        wb = app.books.open(ruta_macro)
        try: ws = wb.sheets[nombre_hoja]; ws.clear()
        except: ws = wb.sheets.add(nombre_hoja)
        current_col = 1
        for df_tabla, titulo in tablas:
            df_tabla = df_tabla.fillna('')
            ws.range((1, current_col)).value = titulo
            try: ws.range((1, current_col)).api.Font.Bold = True
            except: pass
            if not df_tabla.empty:
                ws.range((2, current_col)).options(index=False).value = df_tabla
                current_col += len(df_tabla.columns) + 1
            else:
                ws.range((3, current_col)).value = "Sin datos"; current_col += 2
        ws.autofit(); wb.save()
    except Exception as e: print(f"Error xlwings: {e}")
    finally: 
        try: wb.close(); app.quit()
        except: pass

def generar_reporte_plano(df_mp, df_sistema, writer, config):
    print("Generando Reporte Estadístico...")
    try:
        col_turno = config['col_turno']
        col_plat_sis = config['col_plataforma']
        def garantizar_cols(df, cols):
            for c in cols:
                if c not in df.columns: df[c] = "Sin Dato"
            return df

        
        df_m = garantizar_cols(df_mp.copy(), [col_turno, 'Estado', 'datetime_col', 'monto_col_numeric'])
        df_s = garantizar_cols(df_sistema.copy(), [col_turno, 'Estado', 'datetime_col', 'monto_col_numeric', col_plat_sis])
        
        
        df_m['Plataforma'] = 'Mercado Pago'; df_m['Medio de cobro'] = 'Mercado Pago'
        df_s['Plataforma'] = 'Sistema'; df_s['Medio de cobro'] = df_s[col_plat_sis]

        
        df_m['Monto'] = df_m['monto_col_numeric']
        df_s['Monto'] = df_s['monto_col_numeric']
        
        
        try: df_m['Fecha_Grp'] = df_m['datetime_col'].dt.strftime('%d/%m/%Y')
        except: df_m['Fecha_Grp'] = "Sin Fecha"
        try: df_s['Fecha_Grp'] = df_s['datetime_col'].dt.strftime('%d/%m/%Y')
        except: df_s['Fecha_Grp'] = "Sin Fecha"
        
        
        df_m['Estado_Simple'] = df_m['Estado'].apply(lambda x: 'Conciliado' if x == 'Conciliado' else 'No Conciliado')
        df_s['Estado_Simple'] = df_s['Estado'].apply(lambda x: 'Conciliado' if x == 'Conciliado' else 'No Conciliado')

        cols_req = ['Fecha_Grp', 'TURNO', 'Plataforma', 'Medio de cobro', 'Monto', 'Estado_Simple']
        dfs_to_concat = [d[cols_req] for d in [df_m, df_s] if not d.empty]
        
        if dfs_to_concat:
            df_total = pd.concat(dfs_to_concat, ignore_index=True)
            df_total['Cantidad'] = 1

            df_agg = df_total.groupby(['Fecha_Grp', 'TURNO', 'Plataforma', 'Medio de cobro', 'Estado_Simple']).agg(
                Monto_Total=('Monto', 'sum'), Cantidad_Total=('Cantidad', 'count')
            ).reset_index()

            df_p_m = df_agg.pivot_table(index=['Fecha_Grp', 'TURNO', 'Plataforma', 'Medio de cobro'], columns='Estado_Simple', values='Monto_Total').fillna(0)
            df_p_c = df_agg.pivot_table(index=['Fecha_Grp', 'TURNO', 'Plataforma', 'Medio de cobro'], columns='Estado_Simple', values='Cantidad_Total').fillna(0)
            
            df_p_m.columns = [f"Total Monto {col}" for col in df_p_m.columns]
            df_p_c.columns = [f"Cantidad {col}" for col in df_p_c.columns]
            
            df_reporte = df_p_m.join(df_p_c).reset_index().rename(columns={'Fecha_Grp': 'Fecha'})
            
            for col in ['Total Monto Conciliado', 'Total Monto No Conciliado', 'Cantidad Conciliado', 'Cantidad No Conciliado']:
                if col not in df_reporte.columns: df_reporte[col] = 0
                
            df_reporte['Total Monto'] = df_reporte['Total Monto Conciliado'] + df_reporte['Total Monto No Conciliado']
            df_reporte['Total Cantidad'] = df_reporte['Cantidad Conciliado'] + df_reporte['Cantidad No Conciliado']
            
            df_reporte.fillna(0).to_excel(writer, sheet_name='Reporte_Estadistico_Plano', index=False)
    except Exception as e: print(f"Error Reporte: {e}"); traceback.print_exc()

def generar_hoja_auditoria(df_mp, df_py, df_sistema, writer, config): # <--- AHORA RECIBE df_py
    print("Generando Auditoría...")
    try:
        col_turno = config['col_turno']
        col_plat = config['col_plataforma']
        dfs = []

        # 1. Auditoría MP
        if not df_mp.empty:
            mask_excluir = calcular_mascara_exclusion(df_mp['Clasificacion'])
            dm = df_mp[(df_mp['Estado'] == 'No Conciliado') & (~mask_excluir)].copy()
            dm['Origen'] = 'Mercado Pago'
            dm['Fecha y Hora'] = convertir_a_string_visual(dm['datetime_col'])
            dm = dm.rename(columns={config['col_monto_mp']: 'Monto', config['col_id_mp']: 'ID_Operacion', 'Estado_Auditoria': 'Estado_Explicacion', col_turno: 'Turno'})
            dfs.append(dm)

        # 2. Auditoría PedidosYa (NUEVO)
        if not df_py.empty:
            dpy = df_py[df_py['Estado'] == 'No Conciliado'].copy()
            dpy['Origen'] = 'Pedidos Ya'
            dpy['Fecha y Hora'] = convertir_a_string_visual(dpy['datetime_col'])
            dpy = dpy.rename(columns={config['col_monto_py']: 'Monto', config['col_id_py']: 'ID_Operacion', col_turno: 'Turno'})
            # PY no tiene estado auditoria, pero lo agregamos vacio
            dpy['Estado_Explicacion'] = "" 
            dfs.append(dpy)

        # 3. Auditoría Sistema
        if not df_sistema.empty:
            ds = df_sistema[df_sistema['Estado'] == 'No Conciliado'].copy()
            ds['Origen'] = 'Sistema (' + ds[col_plat].astype(str) + ')'
            ds['Fecha y Hora'] = convertir_a_string_visual(ds['datetime_col'])
            ds = ds.rename(columns={config['col_monto_sis']: 'Monto', config['col_id_sis']: 'ID_Operacion', 'Estado_Auditoria': 'Estado_Explicacion', col_turno: 'Turno'})
            dfs.append(ds)
        
        if dfs:
            df_audit = pd.concat(dfs, ignore_index=True)
            # Unificar columnas para evitar errores
            cols_export = ['Origen', 'Fecha y Hora', 'Turno', 'ID_Operacion', 'Monto', 'Estado_Explicacion']
            df_audit = df_audit[[c for c in cols_export if c in df_audit.columns]]
            df_audit.to_excel(writer, sheet_name='Auditoría_Pendientes', index=False)
    except Exception as e: print(f"Error Auditoria: {e}")

def generar_hoja_tablas_revision_y_exportar(df_mp, df_py, df_sistema, writer, config): 
    print("Generando Tablas Revisión...")

    def preparar_tabla_export(df_source, col_id_primary, titulo, incluir_detalles_cruce=False):
        # 1. Si la tabla está vacía, devolvemos estructura vacía sin protestar
        if df_source.empty:
            cols = ['ID', 'Fecha y Hora', 'Fecha', 'Hora', 'TURNO', 'Monto']
            if incluir_detalles_cruce: cols += ['Tipo Match', 'Plataforma Cruce']
            return pd.DataFrame(columns=cols), titulo
            
        df_temp = df_source.copy()
        
        # Seguridad para columnas faltantes
        if 'datetime_col' not in df_temp.columns: df_temp['datetime_col'] = pd.NaT
        if col_id_primary not in df_temp.columns: df_temp[col_id_primary] = "Sin ID"
        if 'monto_col_numeric' not in df_temp.columns: df_temp['monto_col_numeric'] = 0

        df_temp['ID'] = df_temp[col_id_primary]
        df_temp['Fecha y Hora'] = df_temp['datetime_col']
        
        try: df_temp['Fecha'] = df_temp['datetime_col'].dt.date
        except: df_temp['Fecha'] = pd.NaT
        try: df_temp['Hora'] = df_temp['datetime_col'].dt.strftime('%H:%M:%S')
        except: df_temp['Hora'] = ""
        
        df_temp['Monto'] = df_temp['monto_col_numeric']
        
        cols_finales = ['ID', 'Fecha y Hora', 'Fecha', 'Hora', 'TURNO', 'Monto']
        
        if incluir_detalles_cruce:
            def get_platform_name(row):
                if 'ID Operación MP (Conc.)' in row.index and pd.notna(row['ID Operación MP (Conc.)']): return "Mercado Pago"
                return ""
            df_temp['Plataforma Cruce'] = df_temp.apply(get_platform_name, axis=1)
            cols_finales.extend(['Tipo Match', 'Plataforma Cruce'])

        return df_temp[[c for c in cols_finales if c in df_temp.columns]], titulo

    # --- 1. MP No Conciliado ---
    # BLINDAJE: Verificamos si existe 'Estado'
    if not df_mp.empty and 'Estado' in df_mp.columns:
        mask_excluir = calcular_mascara_exclusion(df_mp['Clasificacion'])
        df_mp_no_conc = df_mp[(df_mp['Estado'] == 'No Conciliado') & (~mask_excluir)]
    else:
        df_mp_no_conc = pd.DataFrame() # Si no hay datos, creamos vacío
        
    t1, h1 = preparar_tabla_export(df_mp_no_conc, config['col_id_mp'], "MP NO CONCILIADO")
    
    # --- 2. PEDIDOS YA No Conciliado (AQUÍ ESTABA TU ERROR) ---
    # BLINDAJE: Verificamos si existe 'Estado' antes de filtrar
    if not df_py.empty and 'Estado' in df_py.columns:
        df_py_no_conc = df_py[df_py['Estado'] == 'No Conciliado']
    else:
        df_py_no_conc = pd.DataFrame() # Esto evita el KeyError
        
    t_py, h_py = preparar_tabla_export(df_py_no_conc, config['col_id_py'], "PEDIDOS YA NO CONCILIADO")

    # --- 3. Sistema (MP) No Conciliado ---
    if not df_sistema.empty and 'Estado' in df_sistema.columns:
        df_sys_mp = df_sistema[(df_sistema[config['col_plataforma']] == config['val_mp']) & (df_sistema['Estado'] == 'No Conciliado')]
    else:
        df_sys_mp = pd.DataFrame()
        
    t3, h3 = preparar_tabla_export(df_sys_mp, config['col_id_sis'], "SISTEMA (MP) NO CONCILIADO")

    # --- 4. Sistema (PY) No Conciliado ---
    if not df_sistema.empty and 'Estado' in df_sistema.columns:
        df_sys_py = df_sistema[(df_sistema[config['col_plataforma']] == config['val_py_sis']) & (df_sistema['Estado'] == 'No Conciliado')]
    else:
        df_sys_py = pd.DataFrame()
        
    t4, h4 = preparar_tabla_export(df_sys_py, config['col_id_sis'], "SISTEMA (PY) NO CONCILIADO")

    # --- 5. Matches Cruzados ---
    if not df_sistema.empty and 'Estado' in df_sistema.columns and 'Tipo Match' in df_sistema.columns:
        df_cruzados = df_sistema[(df_sistema['Estado'] == 'Conciliado') & (df_sistema['Tipo Match'].astype(str).str.contains('Global', case=False, na=False))]
    else:
        df_cruzados = pd.DataFrame()
        
    t5, h5 = preparar_tabla_export(df_cruzados, config['col_id_sis'], "MATCHES CRUZADOS / GLOBALES", True)
    
    # --- 6. Efectivo Conciliado ---
    if not df_sistema.empty and 'Estado' in df_sistema.columns:
        df_efectivo = df_sistema[(df_sistema[config['col_plataforma']] == config['val_efectivo']) & (df_sistema['Estado'] == 'Conciliado')]
    else:
        df_efectivo = pd.DataFrame()
        
    t6, h6 = preparar_tabla_export(df_efectivo, config['col_id_sis'], "EFECTIVO CONCILIADO", True)
    
    # --- 7. Cta Cte Conciliado ---
    if not df_sistema.empty and 'Estado' in df_sistema.columns:
        df_ctacte = df_sistema[(df_sistema[config['col_plataforma']] == config['val_cta_cte']) & (df_sistema['Estado'] == 'Conciliado')]
    else:
        df_ctacte = pd.DataFrame()
        
    t7, h7 = preparar_tabla_export(df_ctacte, config['col_id_sis'], "CTA CTE CONCILIADO", True)

    # Exportar
    tablas = [(t1, h1), (t_py, h_py), (t3, h3), (t4, h4), (t5, h5), (t6, h6), (t7, h7)]
    exportar_tablas_a_xlwings(tablas, FILES_OUTPUT['archivo_macro'], HOJA_NO_CONCILIADO_DESTINO)

def marcar_match(df_p, idx_p, df_s, idx_s, tipo, col_id_p, col_id_s, col_conc_p, col_conc_s):
    df_p.at[idx_p, 'Estado'] = 'Conciliado'
    df_s.at[idx_s, 'Estado'] = 'Conciliado'
    df_p.at[idx_p, 'Tipo Match'] = tipo
    df_s.at[idx_s, 'Tipo Match'] = tipo
    df_p.at[idx_p, col_conc_p] = df_s.at[idx_s, col_id_s]
    df_s.at[idx_s, col_conc_s] = df_p.at[idx_p, col_id_p]

def correr_conciliacion_propinas_desdoblada(df_mp, df_sis, config, indices_usados):
    """
    Regla especial: 1 Venta Sistema (Con Propina) vs 2 Movimientos MP (Cobro + Propina).
    Compara:
      1. Sistema['Propina'] vs MP['Monto'] (Donde Tipo Operacion == 'Propina')
      2. Sistema['A Pagar'] vs MP['Monto'] (Donde Tipo Operacion != 'Propina')
    """
    print("  Fase Especial: Propinas Desdobladas (1 a 2)...")
    
    col_turno = config['col_turno']
    col_plat_sis = config['col_plataforma']
    val_sis = config['val_mp']
    col_propina = config['col_propina_sis']
    col_apag = config['col_a_pagar_sis']
    col_tipo_mp = config['col_tipo_op_mp']
    
    matches_count = 0
    
    # 1. Filtramos Sistema: No conciliados, que sean MP, y que tengan Propina > 0
    mask_sis = (
        (df_sis['Estado'] == 'No Conciliado') & 
        (~df_sis.index.isin(indices_usados)) &
        (df_sis[col_plat_sis] == val_sis) &
        (df_sis[col_propina] > 0)
    )
    
    # Iteramos sobre las ventas candidatas del sistema
    for idx_s, row_s in df_sis[mask_sis].iterrows():
        
        monto_propina = row_s[col_propina]
        monto_neto = row_s[col_apag] # El valor 'A Pagar' (sin propina)
        turno_s = row_s[col_turno]
        fecha_turno_s = row_s['FECHA_TURNO'] if 'FECHA_TURNO' in row_s else pd.NaT
        
        # --- BUSQUEDA PARTE 1: LA PROPINA EN MP ---
        # Buscamos en MP filas NO conciliadas, del mismo turno, que sean TIPO 'Propina' y monto exacto
        mask_mp_prop = (
            (df_mp['Estado'] == 'No Conciliado') &
            (~df_mp.index.isin(indices_usados)) &
            (df_mp[col_turno] == turno_s) &
            (df_mp[config['col_monto_mp']] == monto_propina)
        )
        
        # Filtro de tipo de operación (si existe la columna en el excel)
        if col_tipo_mp in df_mp.columns:
            # Buscamos que diga "Propina" (ignorando mayusculas/minusculas)
            mask_mp_prop = mask_mp_prop & (df_mp[col_tipo_mp].astype(str).str.contains('Propina', case=False, na=False))
        
        # Filtro de fecha turno exacto
        if pd.notna(fecha_turno_s) and 'FECHA_TURNO' in df_mp.columns:
            mask_mp_prop = mask_mp_prop & (df_mp['FECHA_TURNO'] == fecha_turno_s)

        candidatos_propina = df_mp[mask_mp_prop]
        
        if candidatos_propina.empty:
            continue # Si no encontramos la propina, no seguimos buscando el neto
            
        # Tomamos el primer candidato de propina
        idx_mp_prop = candidatos_propina.index[0]
        id_mp_prop = df_mp.at[idx_mp_prop, config['col_id_mp']]
        
        # --- BUSQUEDA PARTE 2: EL NETO (A PAGAR) EN MP ---
        # Buscamos el resto del dinero. NO debe ser tipo Propina.
        mask_mp_neto = (
            (df_mp['Estado'] == 'No Conciliado') &
            (~df_mp.index.isin(indices_usados)) &
            (df_mp.index != idx_mp_prop) & # Que no sea la misma fila que acabamos de encontrar
            (df_mp[col_turno] == turno_s) &
            (df_mp[config['col_monto_mp']] == monto_neto)
        )
        
        if col_tipo_mp in df_mp.columns:
             # Que NO diga Propina (usualmente dice "Cobro" o "Ingreso")
            mask_mp_neto = mask_mp_neto & (~df_mp[col_tipo_mp].astype(str).str.contains('Propina', case=False, na=False))
            
        if pd.notna(fecha_turno_s) and 'FECHA_TURNO' in df_mp.columns:
            mask_mp_neto = mask_mp_neto & (df_mp['FECHA_TURNO'] == fecha_turno_s)
            
        candidatos_neto = df_mp[mask_mp_neto]
        
        if candidatos_neto.empty:
            continue # Tenemos la propina pero no el cobro principal. Abortamos match.
            
        # Tomamos el primer candidato neto
        idx_mp_neto = candidatos_neto.index[0]
        id_mp_neto = df_mp.at[idx_mp_neto, config['col_id_mp']]
        
        # --- ¡MATCH CONFIRMADO! (1 Sistema vs 2 MP) ---
        
        # 1. Marcar Sistema
        df_sis.at[idx_s, 'Estado'] = 'Conciliado'
        df_sis.at[idx_s, 'Tipo Match'] = 'Propina Desdoblada'
        # Guardamos AMBOS IDs de MP en el sistema
        df_sis.at[idx_s, 'ID Operación MP (Conc.)'] = f"{id_mp_neto} + {id_mp_prop}"
        
        # 2. Marcar MP (Propina)
        df_mp.at[idx_mp_prop, 'Estado'] = 'Conciliado'
        df_mp.at[idx_mp_prop, 'Tipo Match'] = 'Propina (Parte)'
        df_mp.at[idx_mp_prop, 'ID Venta Sistema (Conc.)'] = df_sis.at[idx_s, config['col_id_sis']]
        
        # 3. Marcar MP (Neto)
        df_mp.at[idx_mp_neto, 'Estado'] = 'Conciliado'
        df_mp.at[idx_mp_neto, 'Tipo Match'] = 'Propina (Base)'
        df_mp.at[idx_mp_neto, 'ID Venta Sistema (Conc.)'] = df_sis.at[idx_s, config['col_id_sis']]
        
        # Actualizar indices usados
        indices_usados.add(idx_s)
        indices_usados.add(idx_mp_prop)
        indices_usados.add(idx_mp_neto)
        
        matches_count += 1

    if matches_count > 0:
        print(f"    -> {matches_count} matches de PROPINA DESDOBLADA encontrados.")

def correr_conciliacion(df_plat, df_sis, nombre_fase, val_sis, config, indices_usados, ignorar_medio=False):
    col_turno = config['col_turno']; col_plat_sis = config['col_plataforma']
    print(f"  Fase {nombre_fase}...")
    
    # 1. DETECTAR QUÉ COLUMNA DE ID USAR (MP o PY)
    # Si la columna de MP existe en este archivo, usamos esa. Si no, probamos la de PY.
    col_id_mp_key = config['col_id_mp']
    col_id_py_key = config.get('col_id_py', 'Nro de pedido')
    
    if col_id_mp_key in df_plat.columns:
        col_id_p_uso = col_id_mp_key
    elif col_id_py_key in df_plat.columns:
        col_id_p_uso = col_id_py_key
    else:
        # Si no encuentra ninguna (caso raro), usa una genérica para no romper, aunque no grabará ID
        col_id_p_uso = 'ID_Generico' 

    for regla, min_tol, money_tol in REGLAS_CONCILIACION:
        matches = 0; is_date_only_match = regla.startswith("R3:")
        
        for i_p in df_plat.index:
            if df_plat.at[i_p, 'Estado'] != 'No Conciliado': continue
            
            t_p = df_plat.at[i_p, 'datetime_col']
            m_p = df_plat.at[i_p, 'monto_col_numeric']
            turno_p = df_plat.at[i_p, col_turno]
            
            fecha_turno_p = df_plat.at[i_p, 'FECHA_TURNO'] if 'FECHA_TURNO' in df_plat.columns else pd.NaT

            if pd.isna(t_p) or pd.isna(m_p): continue

            mask = ~df_sis.index.isin(indices_usados) & \
                   (df_sis['Estado'] == 'No Conciliado') & \
                   (df_sis[col_turno] == turno_p)
            
            if pd.notna(fecha_turno_p) and 'FECHA_TURNO' in df_sis.columns:
                mask = mask & (df_sis['FECHA_TURNO'] == fecha_turno_p)

            if not ignorar_medio: mask = mask & (df_sis[col_plat_sis] == val_sis)
            
            candidatos = df_sis[mask]
            
            if is_date_only_match:
                if t_p is pd.NaT: continue
                match = candidatos[
                    (candidatos['datetime_col'].dt.date == t_p.date()) & 
                    (candidatos['monto_col_numeric'].between(m_p - money_tol, m_p + money_tol))
                ]
            else:
                match = candidatos[
                    (candidatos['datetime_col'].between(t_p - pd.Timedelta(minutes=min_tol), t_p + pd.Timedelta(minutes=min_tol))) & 
                    (candidatos['monto_col_numeric'].between(m_p - money_tol, m_p + money_tol))
                ]
            
            if not match.empty:
                i_s = match.index[0]
                
                col_conc_p = 'ID Venta Sistema (Conc.)'
                col_conc_s = 'ID Operación MP (Conc.)' # Usamos esta columna en el sistema para guardar el ID de cualquiera de las dos apps
                
                # CORRECCIÓN AQUÍ: Usamos col_id_p_uso en vez de config['col_id_mp']
                marcar_match(df_plat, i_p, df_sis, i_s, f"{nombre_fase} ({regla})", col_id_p_uso, config['col_id_sis'], col_conc_p, col_conc_s)
                
                indices_usados.add(i_s); matches += 1
                
        if matches > 0: print(f"    -> {matches} matches con regla: {regla}")

if __name__ == "__main__":
    print("=================================================================")
    print("   ASISTENTE DE CONCILIACIÓN (V49 - FINAL FULL DEFINITIVA)")
    print("=================================================================\n")

    ruta_turnos = seleccionar_archivo("Archivo de TURNOS")
    ruta_ventas = seleccionar_archivo("Reporte de VENTAS (Sistema)")
    ruta_mp = seleccionar_archivo("Reporte de MERCADO PAGO")
    ruta_pedidos_ya = seleccionar_archivo("Archivo de PEDIDOS YA") # <--- NUEVO
    ruta_comandas = seleccionar_archivo("Archivo de COMANDAS / DEVOLUCIONES")
    ruta_caja_adicion = seleccionar_archivo("Archivo CAJA ADICION")
    ruta_macro = seleccionar_archivo("Archivo MACRO EXCEL")
    output_folder = seleccionar_carpeta("Carpeta donde guardar el REPORTE FINAL")
    
    FILES_OUTPUT['turnos_proc'] = os.path.join(output_folder, "Turnos_Procesados.xlsx")
    FILES_OUTPUT['informe_gral'] = os.path.join(output_folder, "informes_chimbas.xlsx")
    FILES_OUTPUT['reporte_final'] = os.path.join(output_folder, "Resultado_Conciliacion_chimbas.xlsx")
    FILES_OUTPUT['archivo_macro'] = ruta_macro

    dict_clasificaciones_previas = {}
    if os.path.exists(FILES_OUTPUT['reporte_final']):
        print(">>> Detectado archivo previo de conciliación. Guardando clasificaciones manuales...")
        try:
            df_old = pd.read_excel(FILES_OUTPUT['reporte_final'], sheet_name="MP Conciliado")
            col_clasif = 'Clasificacion'
            col_id_mp = CONFIG_CONCILIACION['col_id_mp']
            if col_clasif in df_old.columns and col_id_mp in df_old.columns:
                df_con_datos = df_old[df_old[col_clasif].notna() & (df_old[col_clasif].astype(str).str.strip() != '')].copy()
                if not df_con_datos.empty:
                    df_con_datos[col_id_mp] = df_con_datos[col_id_mp].astype(str).str.replace(r'\.0$', '', regex=True)
                    dict_clasificaciones_previas = df_con_datos.set_index(col_id_mp)[col_clasif].to_dict()
                    print(f"    -> Se recuperaron {len(dict_clasificaciones_previas)} clasificaciones manuales.")
        except Exception as e:
            print(f"    [AVISO] No se pudo leer el archivo previo: {e}")

    print("\n--- INICIANDO PROCESO ---")
    try:
        print(">>> FASE 1: PROCESAMIENTO")
        df_turnos = procesar_archivo_turnos(ruta_turnos)
        if df_turnos.empty: raise Exception("Error procesando Turnos.")
        
        # Exportar turnos incluyendo columnas nuevas para control
        with pd.ExcelWriter(FILES_OUTPUT['turnos_proc']) as w: 
            df_turnos.to_excel(w, index=False)
            print("    -> Archivo 'Turnos_Procesados.xlsx' actualizado (Todas las columnas incluidas).")

        

        # 2. SISTEMA (PRINCIPAL)
        try:
            print("Procesando Ventas Sistema...")
            df_v = pd.read_excel(ruta_ventas, dtype={'FechaCierre': str, 'Comanda': str})
            
            if 'FechaCierre' in df_v.columns:
                df_v['FechaCierre'] = df_v['FechaCierre'].astype(str).str.replace('"', '', regex=False).str.strip()
                df_v['Fecha_DT'] = normalizar_fecha_argentina(df_v['FechaCierre']) 
                df_v['Fecha_Visual'] = convertir_a_string_visual(df_v['Fecha_DT']) 
            
            # --- CORRECCIÓN: LÓGICA ORIGINAL DE DINAMIZACIÓN ---
            # 1. Definimos las columnas que deben mantenerse FIJAS (no convertirse en filas)
            cols_id = ["FechaCierre", "Fecha_DT", "Fecha_Visual", "Comanda", "Pago", "Total", "Descuentos", "A Pagar", "Propina", "Pagos", "Boleta","Mesa"]
            # Seleccionamos solo las que realmente vienen en el Excel
            vars_id_existentes = [c for c in cols_id if c in df_v.columns]
            
            # 2. Hacemos el melt respetando esas columnas
            df_v = pd.melt(df_v, id_vars=vars_id_existentes, var_name="Atributo", value_name="Valor")
            
            # 3. Limpieza de valores
            df_v['Valor'] = pd.to_numeric(df_v['Valor'], errors='coerce')
            # Asegurar que Propina y A Pagar sean numéricos para la conciliación avanzada
            if 'Propina' in df_v.columns: df_v['Propina'] = pd.to_numeric(df_v['Propina'], errors='coerce').fillna(0)
            if 'A Pagar' in df_v.columns: df_v['A Pagar'] = pd.to_numeric(df_v['A Pagar'], errors='coerce').fillna(0)
            df_v = df_v[df_v['Valor'] != 0].dropna(subset=['Valor'])
            
            # 4. Eliminamos filas basura que se generan si el Excel trae columnas extras
            df_v = df_v[~df_v['Atributo'].isin(['Caja', '#Boleta', 'Total', 'A Pagar'])]
            
            # 5. Renombrado y Asignación de Turno
            df_v = df_v.rename(columns={"Fecha_Visual": "Fecha", "Comanda": "ID de venta", "Valor": "Monto Bruto pago", "Atributo": "Medio de cobro"})
            df_v = asignar_turno_desde_excel(df_v, "Fecha_DT", df_turnos) 
            
        except Exception as e: 
            print(f"Error Ventas: {e}")
            traceback.print_exc()
            df_v = pd.DataFrame()
        # 3. MERCADO PAGO (PRINCIPAL - Para Conciliación)
        try:
            print("Procesando Mercado Pago (Conciliación)...")
            df_mp = pd.read_excel(ruta_mp)
            df_mp.columns = df_mp.columns.astype(str).str.strip().str.upper()

            # Exclusión de POS específico
            serial_excluir = 'SMARTPOS1493608722'
            if 'NÚMERO DE SERIE DEL LECTOR (S/N)' in df_mp.columns:
                df_mp = df_mp[df_mp['NÚMERO DE SERIE DEL LECTOR (S/N)'].astype(str).str.strip() != serial_excluir]

            col_fecha_mp = 'FECHA DE ORIGEN'
            if col_fecha_mp in df_mp.columns:
                df_mp['FECHA_DT'] = parsear_fecha_mp_iso(df_mp[col_fecha_mp])
                df_mp[col_fecha_mp] = convertir_a_string_visual(df_mp['FECHA_DT'])
            else: df_mp['FECHA_DT'] = pd.NaT
            
            if 'VALOR DE LA COMPRA' in df_mp.columns: 
                df_mp['VALOR DE LA COMPRA'] = pd.to_numeric(df_mp['VALOR DE LA COMPRA'], errors='coerce').fillna(0)
            
            mask_positivo = df_mp['VALOR DE LA COMPRA'] > 0
            df_mp = df_mp[mask_positivo & (df_mp['MEDIO DE PAGO'].astype(str).str.lower() != "nan")]
            if 'ID DE OPERACIÓN EN MERCADO PAGO' in df_mp.columns: 
                df_mp['ID DE OPERACIÓN EN MERCADO PAGO'] = df_mp['ID DE OPERACIÓN EN MERCADO PAGO'].astype(str).str.replace(r'\.0$', '', regex=True)
            df_mp = asignar_turno_desde_excel(df_mp, "FECHA_DT", df_turnos)
        except Exception as e: print(f"Error MP: {e}"); df_mp = pd.DataFrame()

        # 4. MERCADO PAGO (SECUNDARIO - Pagos Negativos)
        df_mp_negativos = obtener_df_pagos_mp_negativos(ruta_mp, df_turnos)

        # -------------------------------------------------------------------------
        # NUEVO: PROCESAR PEDIDOS YA
        # -------------------------------------------------------------------------
        df_py_clean, df_py_cancelados = procesar_pedidos_ya_retorno(ruta_pedidos_ya, df_turnos)

        # GUARDAR INFO GRAL CON FORMATO
        with pd.ExcelWriter(FILES_OUTPUT['informe_gral'], engine='xlsxwriter', datetime_format='dd/mm/yyyy hh:mm:ss') as writer:
            if not df_v.empty: df_v.to_excel(writer, sheet_name="Ventas_Sistema", index=False)
            if not df_mp.empty: df_mp.to_excel(writer, sheet_name="Ventas_MP", index=False)
            
            # NUEVO: GUARDAR LA PESTAÑA LIMPIA DE PEDIDOS YA
            if not df_py_clean.empty: 
                df_py_clean.to_excel(writer, sheet_name="Ventas_PY", index=False)

            if not df_mp_negativos.empty: 
                df_mp_negativos.to_excel(writer, sheet_name="Pagos_MP_Para_Copiar", index=False)
                # Formato visual
                workbook = writer.book
                worksheet = writer.sheets["Pagos_MP_Para_Copiar"]
                fmt_fh = workbook.add_format({'num_format': 'dd/mm/yyyy hh:mm:ss'})
                worksheet.set_column('A:A', 22, fmt_fh)
        
        # PROCESAR MACROS (PRINCIPAL - Comandas y Caja)
        procesar_comandas(ruta_comandas, FILES_OUTPUT['archivo_macro'], HOJA_COMANDAS_DESTINO, df_turnos)
        procesar_caja_adicion(ruta_caja_adicion, FILES_OUTPUT['archivo_macro'], HOJA_CAJA_ADICION_DESTINO, df_turnos)
        
        # NUEVO: GUARDAR LOS CANCELADOS DE PY EN EL MACRO
        exportar_a_macro_simple(df_py_cancelados, FILES_OUTPUT['archivo_macro'], HOJA_CANCELADOS_PY_DESTINO)
        
        print(">>> FASE 1 OK.")
    except Exception as e: print(f"ERROR FASE 1: {e}"); traceback.print_exc(); sys.exit(1)

    try:
        print("\n>>> FASE 2: CONCILIACIÓN")
        cfg = CONFIG_CONCILIACION
        xls_obj = pd.ExcelFile(FILES_OUTPUT['informe_gral'])
        def leer_hoja_segura(xls, nombre): return pd.read_excel(xls, sheet_name=nombre) if nombre in xls.sheet_names else pd.DataFrame()

        # LEEMOS LAS 3 HOJAS: MP, PY (NUEVO) Y SISTEMA
        df_mp = leer_hoja_segura(xls_obj, "Ventas_MP")
        df_py = leer_hoja_segura(xls_obj, "Ventas_PY") # <--- NUEVO
        df_sis = leer_hoja_segura(xls_obj, "Ventas_Sistema")

        # Normalizamos Fechas de Turno si existen
        if 'FECHA_TURNO' in df_mp.columns: df_mp['FECHA_TURNO'] = pd.to_datetime(df_mp['FECHA_TURNO'], dayfirst=True, errors='coerce')
        if 'FECHA_TURNO' in df_py.columns: df_py['FECHA_TURNO'] = pd.to_datetime(df_py['FECHA_TURNO'], dayfirst=True, errors='coerce')
        if 'FECHA_TURNO' in df_sis.columns: df_sis['FECHA_TURNO'] = pd.to_datetime(df_sis['FECHA_TURNO'], dayfirst=True, errors='coerce')

        # PREPARAR DATAFRAMES (MP, PY, SISTEMA)
        # Aquí configuramos qué columna de fecha y monto usa cada plataforma
        lista_dfs = [
            (df_mp, cfg['col_fecha_mp'], cfg['col_monto_mp']),
            (df_py, cfg['col_fecha_py'], cfg['col_monto_py']), # <--- NUEVO
            (df_sis, 'Fecha', cfg['col_monto_sis'])
        ]

        for df, c_fecha, c_monto in lista_dfs:
            if not df.empty:
                if cfg['col_turno'] not in df.columns: df[cfg['col_turno']] = "Sin Turno"
                df['datetime_col'] = pd.to_datetime(df.get(c_fecha, pd.Series()), dayfirst=True, errors='coerce') 
                df['monto_col_numeric'] = pd.to_numeric(df.get(c_monto, pd.Series()), errors='coerce')
                df['Estado'] = 'No Conciliado'; df['Tipo Match'] = pd.NA
                
                # Solo sistema tiene medio de cobro explicito, para los otros generamos Clasificacion vacia
                if df is df_sis: 
                    df[cfg['col_plataforma']] = df.get(cfg['col_plataforma'], pd.Series()).fillna("-").astype(str)
                else:
                    df['Clasificacion'] = ""

        indices_usados = set()
        
        # --- 1. CONCILIACION MERCADO PAGO ---
        if not df_mp.empty and not df_sis.empty: 
            correr_conciliacion(df_mp, df_sis, "F1: MP", cfg['val_mp'], cfg, indices_usados)
            correr_conciliacion_propinas_desdoblada(df_mp, df_sis, cfg, indices_usados)
        
        # --- 2. CONCILIACION PEDIDOS YA (NUEVO) ---
        if not df_py.empty and not df_sis.empty:
            # Cruza: df_py vs df_sis filtrando por "Pedidos Ya" en el sistema
            correr_conciliacion(df_py, df_sis, "F2: PedidosYa", cfg['val_py_sis'], cfg, indices_usados)

        # --- 3. CONCILIACION RESTANTE (Efectivo, Cta Cte, MP Global) ---
        if not df_mp.empty and not df_sis.empty: 
            correr_conciliacion(df_mp, df_sis, "F3: MP vs Efec", "Efectivo", cfg, indices_usados)
            correr_conciliacion(df_mp, df_sis, "F4: MP vs Cta Cte", cfg['val_cta_cte'], cfg, indices_usados)
            correr_conciliacion(df_mp, df_sis, "F5: MP Global", None, cfg, indices_usados, ignorar_medio=True)
        
        # RECUPERAR CLASIFICACIONES MANUALES (Solo para MP por ahora)
        if not df_mp.empty and dict_clasificaciones_previas:
            if cfg['col_id_mp'] in df_mp.columns:
                df_mp['Clasificacion'] = df_mp[cfg['col_id_mp']].astype(str).str.replace(r'\.0$', '', regex=True).map(dict_clasificaciones_previas)

        with pd.ExcelWriter(FILES_OUTPUT['reporte_final']) as writer:
            # Generamos reportes estadísticos y auditoría
            # CORRECCIÓN: Cambiamos df_sistema por df_sis
            generar_reporte_plano( df_mp, df_sis, writer, cfg)
            generar_hoja_auditoria(df_mp, df_py, df_sis, writer, cfg) 
            
            # Función auxiliar para limpiar columnas técnicas antes de exportar
            def clean_export(df):
                if df.empty: return df
                if 'FECHA_TURNO' in df.columns:
                     df['Fecha (Solo)'] = df['FECHA_TURNO'].dt.strftime('%d/%m/%Y').fillna(df['datetime_col'].dt.strftime('%d/%m/%Y'))
                else:
                     df['Fecha (Solo)'] = df['datetime_col'].dt.strftime('%d/%m/%Y')
                return df.drop(columns=['datetime_col', 'monto_col_numeric', 'Estado_Auditoria', 'FECHA_DT', 'Fecha_DT', 'FECHA_TURNO'], errors='ignore')

            # Exportamos las hojas finales
            if not df_mp.empty: clean_export(df_mp).to_excel(writer, sheet_name="MP Conciliado", index=False)
            if not df_py.empty: clean_export(df_py).to_excel(writer, sheet_name="PY Conciliado", index=False)
            if not df_sis.empty: clean_export(df_sis).to_excel(writer, sheet_name="Sistema Conciliado", index=False)

            # Tablas de revision
            # CORRECCIÓN: Cambiamos df_sistema por df_sis
            generar_hoja_tablas_revision_y_exportar(df_mp, df_py, df_sis, writer, cfg)
            
        print(">>> PROCESO FINALIZADO.")
    except Exception as e: print(f"ERROR FASE 2: {e}"); traceback.print_exc()
    input("Presiona ENTER para salir...")

   ASISTENTE DE CONCILIACIÓN (V49 - FINAL FULL DEFINITIVA)

--> Por favor, selecciona el archivo: Archivo de TURNOS
    OK: Turnos chimbas.xlsx
--> Por favor, selecciona el archivo: Reporte de VENTAS (Sistema)
    OK: Ventas Chimbas.xlsx
--> Por favor, selecciona el archivo: Reporte de MERCADO PAGO
    OK: settlement_v2-2932664883-manual-2026-02-04-070758.xlsx
--> Por favor, selecciona el archivo: Archivo de PEDIDOS YA
    OK: orderDetails - 2026-02-04T081315.261.xlsx
--> Por favor, selecciona el archivo: Archivo de COMANDAS / DEVOLUCIONES
    OK: Devolucion Chimbas.xlsx
--> Por favor, selecciona el archivo: Archivo CAJA ADICION
    OK: Caja Adicion Chimbas.xlsx
--> Por favor, selecciona el archivo: Archivo MACRO EXCEL
    OK: ARQUEO CHIMBAS (1) (1).xlsm
--> Por favor, selecciona la CARPETA DE DESTINO.
    OK: Carpeta G:/Mi unidad/Modelo de automatizaciones/Chimbas Mendoza/03 02 2026
>>> Detectado archivo previo de conciliación. Guardando clasificaciones manuales...

--- INICIANDO PROC